In [1]:
# ============================================
# MODELAGEM - PREVISÃO DE CONSUMO ENERGÉTICO
# ============================================

import pandas as pd
import numpy as np

# Carregar dataset com features
df = pd.read_csv('../data/processed/data_with_features.csv')

print("="*70)
print("📊 DATASET CARREGADO")
print("="*70)
print(f"\nLinhas: {len(df):,}")
print(f"Colunas: {len(df.columns)}")

print("\nColunas disponíveis:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

📊 DATASET CARREGADO

Linhas: 35,041
Colunas: 15

Colunas disponíveis:
   1. Date_Time
   2. Usage_kWh
   3. Lagging_Current_Reactive.Power_kVarh
   4. Lagging_Current_Power_Factor
   5. WeekStatus
   6. Day_Of_Week
   7. Load_Type
   8. Hour
   9. Month
  10. DayOfMonth
  11. is_peak_operational
  12. hour_sin
  13. hour_cos
  14. load_type_encoded
  15. is_weekend


In [2]:
df.head()

,Date_Time,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Lagging_Current_Power_Factor,WeekStatus,Day_Of_Week,Load_Type,Hour,Month,DayOfMonth,is_peak_operational,hour_sin,hour_cos,load_type_encoded,is_weekend
0,2018-01-01 00:15:00,3.17,2.95,73.21,Weekday,Monday,Light_Load,0,1,1,0,0.000000,1.000000,8.625959,0
1,2018-01-01 00:30:00,4.00,4.46,66.77,Weekday,Monday,Light_Load,0,1,1,0,0.000000,1.000000,8.625959,0
2,2018-01-01 00:45:00,3.24,3.28,70.28,Weekday,Monday,Light_Load,0,1,1,0,0.000000,1.000000,8.625959,0
3,2018-01-01 01:00:00,3.31,3.56,68.09,Weekday,Monday,Light_Load,1,1,1,0,0.258819,0.965926,8.625959,0
4,2018-01-01 01:15:00,3.82,4.50,64.72,Weekday,Monday,Light_Load,1,1,1,0,0.258819,0.965926,8.625959,0


In [3]:
from sklearn.model_selection import train_test_split

print("="*70)
print("🔒 PREVENINDO DATA LEAKAGE")
print("="*70)

# ============================================
# PASSO 1: Separar TARGET e FEATURES (SEM load_type_encoded ainda)
# ============================================

TARGET = 'Usage_kWh'

# Features ORIGINAIS (sem load_type_encoded)
features_originais = [
    'Lagging_Current_Reactive.Power_kVarh',
    'Lagging_Current_Power_Factor',
    'Hour',
    'Month',
    'DayOfMonth',
    'is_peak_operational',
    'hour_sin',
    'hour_cos',
    'is_weekend',
    'Load_Type'  # ← Vamos manter texto por enquanto
]

X_original = df[features_originais]
y = df[TARGET]

# ============================================
# PASSO 2: DIVIDIR primeiro (ANTES de fazer encoding)
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X_original, y,
    test_size=0.2,
    random_state=42
)

print(f"\n✅ Dados divididos:")
print(f"   Treino: {len(X_train):,} linhas")
print(f"   Teste:  {len(X_test):,} linhas")

# ============================================
# PASSO 3: Calcular encoding APENAS com TREINO
# ============================================

# Calcular médias usando APENAS dados de treino
train_data = pd.concat([X_train, y_train], axis=1)
medias_treino = train_data.groupby('Load_Type')['Usage_kWh'].mean()

print(f"\n📊 Médias calculadas (APENAS treino):")
for load_type, media in medias_treino.items():
    print(f"   {load_type:15s}: {media:.2f} kWh")

# ============================================
# PASSO 4: Aplicar encoding em TREINO e TESTE
# ============================================

# Treino
X_train['load_type_encoded'] = X_train['Load_Type'].map(medias_treino)

# Teste (usa encoding calculado no treino!)
X_test['load_type_encoded'] = X_test['Load_Type'].map(medias_treino)

# Remover coluna Load_Type (não precisamos mais)
X_train = X_train.drop('Load_Type', axis=1)
X_test = X_test.drop('Load_Type', axis=1)

print(f"\n✅ Encoding aplicado SEM data leakage!")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_test shape:  {X_test.shape}")


🔒 PREVENINDO DATA LEAKAGE

✅ Dados divididos:
   Treino: 28,032 linhas
   Teste:  7,009 linhas

📊 Médias calculadas (APENAS treino):
   Light_Load     : 8.61 kWh
   Maximum_Load   : 59.32 kWh
   Medium_Load    : 38.32 kWh

✅ Encoding aplicado SEM data leakage!
   X_train shape: (28032, 10)
   X_test shape:  (7009, 10)


#Primeiro Modelo de Regressão Linear simples (Colocando uma Linha de base)

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("="*70)
print("🤖 MODELO 1: LINEAR REGRESSION (Baseline)")
print("="*70)

#treino
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
#previsões
y_pred = model_lr.predict(X_test)

#métricas
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_absolute_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("\n📊 RESULTADOS:")
print(f"   MAE:  {mae:.2f} kWh")
print(f"   RMSE: {rmse:.2f} kWh")
print(f"   R²:   {r2:.4f}")
print(f"   MAPE: {mape:.2f}%")

print("\n✅ Baseline estabelecido!")

🤖 MODELO 1: LINEAR REGRESSION (Baseline)

📊 RESULTADOS:
   MAE:  7.82 kWh
   RMSE: 2.80 kWh
   R²:   0.8924
   MAPE: 84.72%

✅ Baseline estabelecido!


#Modelo 2 (Random Forest)

In [5]:
from sklearn.ensemble import RandomForestRegressor

print("="*70)
print("🌳 MODELO 2: RANDOM FOREST")
print("="*70)

# Treinar
model_rf = RandomForestRegressor(
    n_estimators=100,    # 100 árvores
    random_state=42,
    n_jobs=-1            # Usar todos os cores
)

print("Treinando Random Forest...")
model_rf.fit(X_train, y_train)

# Prever
y_pred_rf = model_rf.predict(X_test)

# Métricas
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)
mape_rf = np.mean(np.abs((y_test - y_pred_rf) / y_test)) * 100

print("\n📊 RESULTADOS:")
print(f"   MAE:  {mae_rf:.2f} kWh")
print(f"   RMSE: {rmse_rf:.2f} kWh")
print(f"   R²:   {r2_rf:.4f}")
print(f"   MAPE: {mape_rf:.2f}%")

# Comparar com baseline
print(f"\n📈 MELHORIA vs Baseline:")
print(f"   MAE:  {((mae - mae_rf)/mae)*100:+.1f}%")
print(f"   R²:   {((r2_rf - r2)/r2)*100:+.1f}%")

🌳 MODELO 2: RANDOM FOREST
Treinando Random Forest...

📊 RESULTADOS:
   MAE:  0.44 kWh
   RMSE: 1.34 kWh
   R²:   0.9984
   MAPE: 3.64%

📈 MELHORIA vs Baseline:
   MAE:  +94.4%
   R²:   +11.9%


achei meio suspeito o resultado, vou verificar overfitting ou outros parâmetros

In [6]:
print("="*70)
print("🔍 VERIFICANDO OVERFITTING")
print("="*70)

# Calcular score no TREINO
y_train_pred_rf = model_rf.predict(X_train)
r2_train_rf = r2_score(y_train, y_train_pred_rf)
mae_train_rf = mean_absolute_error(y_train, y_train_pred_rf)

print("\n🌳 RANDOM FOREST:")
print(f"   TREINO - R²:  {r2_train_rf:.4f} | MAE: {mae_train_rf:.2f}")
print(f"   TESTE  - R²:  {r2_rf:.4f} | MAE: {mae_rf:.2f}")
print(f"   GAP    - R²:  {r2_train_rf - r2_rf:.4f}")

if r2_train_rf - r2_rf > 0.05:
    print("   ⚠️ POSSÍVEL OVERFITTING! Gap > 5%")
elif r2_train_rf > 0.99:
    print("   🚨 SUSPEITO! R² treino muito alto (>99%)")
else:
    print("   ✅ Parece OK!")

🔍 VERIFICANDO OVERFITTING

🌳 RANDOM FOREST:
   TREINO - R²:  0.9996 | MAE: 0.20
   TESTE  - R²:  0.9984 | MAE: 0.44
   GAP    - R²:  0.0011
   🚨 SUSPEITO! R² treino muito alto (>99%)


Fazendo uma validação cruzada

In [7]:
from sklearn.model_selection import cross_val_score

print("="*70)
print("🔄 VALIDAÇÃO CRUZADA (5-Fold)")
print("="*70)

# Fazer validação cruzada com 5 splits
cv_scores = cross_val_score(
    model_rf,           # Modelo
    X_train,            # Features de treino
    y_train,            # Target de treino
    cv=5,               # 5 folds
    scoring='r2',       # Métrica
    n_jobs=-1
)

print("\n📊 RESULTADOS DOS 5 FOLDS:")
for i, score in enumerate(cv_scores, 1):
    print(f"   Fold {i}: R² = {score:.4f}")

print(f"\n📈 ESTATÍSTICAS:")
print(f"   Média:        {cv_scores.mean():.4f}")
print(f"   Desvio padrão: {cv_scores.std():.4f}")
print(f"   Min:          {cv_scores.min():.4f}")
print(f"   Max:          {cv_scores.max():.4f}")

# Interpretar
if cv_scores.std() > 0.05:
    print("\n⚠️ ALTA VARIÂNCIA! Modelo instável entre folds")
elif cv_scores.mean() > 0.99:
    print("\n🚨 MÉDIA MUITO ALTA! Possível data leakage ou overfitting")
else:
    print("\n✅ Validação cruzada OK!")

🔄 VALIDAÇÃO CRUZADA (5-Fold)

📊 RESULTADOS DOS 5 FOLDS:
   Fold 1: R² = 0.9972
   Fold 2: R² = 0.9975
   Fold 3: R² = 0.9973
   Fold 4: R² = 0.9977
   Fold 5: R² = 0.9975

📈 ESTATÍSTICAS:
   Média:        0.9974
   Desvio padrão: 0.0002
   Min:          0.9972
   Max:          0.9977

🚨 MÉDIA MUITO ALTA! Possível data leakage ou overfitting


verificando novamente as features

Acabei percebendo que basicamente temos a potência reativa e o power factor nos dados, com isso da pra calcular a potência ativa que é o target.

In [8]:
# Correlação direta dessa feature com o target
corr_reactive = df['Lagging_Current_Reactive.Power_kVarh'].corr(df['Usage_kWh'])

print("="*70)
print("🔍 INVESTIGANDO A FEATURE DOMINANTE")
print("="*70)
print(f"\nCorrelação Lagging_Reactive vs Usage_kWh: {corr_reactive:.4f}")

if abs(corr_reactive) > 0.95:
    print("🚨 CORRELAÇÃO EXTREMA! Quase linear!")
    print("   Essa variável 'entrega' demais a resposta")

🔍 INVESTIGANDO A FEATURE DOMINANTE

Correlação Lagging_Reactive vs Usage_kWh: 0.8962


### ⚠️ Nota sobre Performance

O modelo alcança R² > 99% devido à alta correlação entre 
Potência Reativa e Consumo (r=0.896).

**Em produção:** Requer medição em tempo real de sensores.
**Trade-off:** Alta acurácia vs dependência de hardware.

In [9]:
# XGBoost e LightGBM juntos!
import xgboost as xgb
import lightgbm as lgb

print("="*70)
print("🚀 MODELOS 3 e 4: XGBoost e LightGBM")
print("="*70)

# XGBoost
print("\n[1/2] Treinando XGBoost...")
model_xgb = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_xgb.fit(X_train, y_train)
y_pred_xgb = model_xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"   XGBoost - MAE: {mae_xgb:.2f} | R²: {r2_xgb:.4f}")

# LightGBM
print("\n[2/2] Treinando LightGBM...")
model_lgb = lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
model_lgb.fit(X_train, y_train)
y_pred_lgb = model_lgb.predict(X_test)

mae_lgb = mean_absolute_error(y_test, y_pred_lgb)
r2_lgb = r2_score(y_test, y_pred_lgb)

print(f"   LightGBM - MAE: {mae_lgb:.2f} | R²: {r2_lgb:.4f}")

print("\n✅ Todos os 4 modelos treinados!")

🚀 MODELOS 3 e 4: XGBoost e LightGBM

[1/2] Treinando XGBoost...
   XGBoost - MAE: 0.78 | R²: 0.9975

[2/2] Treinando LightGBM...
   LightGBM - MAE: 0.85 | R²: 0.9971

✅ Todos os 4 modelos treinados!


In [10]:
print("="*70)
print("📊 COMPARAÇÃO DE TODOS OS MODELOS")
print("="*70)

# Criar tabela de resultados
resultados = pd.DataFrame({
    'Modelo': ['Linear Regression', 'Random Forest', 'XGBoost', 'LightGBM'],
    'MAE': [mae, mae_rf, mae_xgb, mae_lgb],
    'RMSE': [rmse, rmse_rf, np.sqrt(mean_squared_error(y_test, y_pred_xgb)), 
             np.sqrt(mean_squared_error(y_test, y_pred_lgb))],
    'R²': [r2, r2_rf, r2_xgb, r2_lgb],
    'MAPE': [mape, mape_rf, 
             np.mean(np.abs((y_test - y_pred_xgb) / y_test)) * 100,
             np.mean(np.abs((y_test - y_pred_lgb) / y_test)) * 100]
}).sort_values('R²', ascending=False)

print("\n" + resultados.to_string(index=False))

# Identificar melhor modelo
melhor_idx = resultados['R²'].idxmax()
melhor_modelo = resultados.loc[melhor_idx, 'Modelo']
print(f"\n🏆 MELHOR MODELO: {melhor_modelo}")
print(f"   R²: {resultados.loc[melhor_idx, 'R²']:.4f}")
print(f"   MAE: {resultados.loc[melhor_idx, 'MAE']:.2f} kWh")

📊 COMPARAÇÃO DE TODOS OS MODELOS

           Modelo      MAE     RMSE       R²      MAPE
    Random Forest 0.437497 1.338264 0.998428  3.635764
          XGBoost 0.775535 1.685621 0.997507  6.998278
         LightGBM 0.852655 1.807332 0.997134  8.031764
Linear Regression 7.817396 2.795961 0.892350 84.717685

🏆 MELHOR MODELO: Random Forest
   R²: 0.9984
   MAE: 0.44 kWh


In [11]:
import pickle

print("="*70)
print("💾 SALVANDO MELHOR MODELO")
print("="*70)

# Salvar Random Forest (melhor modelo)
with open('../models/best_model_rf.pkl', 'wb') as f:
    pickle.dump(model_rf, f)

print("✅ Modelo salvo em: models/best_model_rf.pkl")
print(f"   Modelo: Random Forest")
print(f"   R²: {r2_rf:.4f}")
print(f"   MAE: {mae_rf:.2f} kWh")

# Salvar também a tabela de resultados
resultados.to_csv('../reports/model_comparison.csv', index=False)
print("\n✅ Comparação salva em: reports/model_comparison.csv")

💾 SALVANDO MELHOR MODELO
✅ Modelo salvo em: models/best_model_rf.pkl
   Modelo: Random Forest
   R²: 0.9984
   MAE: 0.44 kWh

✅ Comparação salva em: reports/model_comparison.csv


#Fazendo teste do modelo em exemplos 

In [13]:
print("="*70)
print("🎯 PREVISÕES EM EXEMPLOS REAIS")
print("="*70)

# Pegar 10 exemplos aleatórios do conjunto de teste
np.random.seed(42)
indices_exemplo = np.random.choice(X_test.index, size=10, replace=False)

# Fazer previsões
exemplos = X_test.loc[indices_exemplo].copy()
y_real = y_test.loc[indices_exemplo]
y_previsto = model_rf.predict(exemplos)

# Criar DataFrame para visualização
comparacao = pd.DataFrame({
    'Real (kWh)': y_real.values,
    'Previsto (kWh)': y_previsto,
    'Erro (kWh)': y_real.values - y_previsto,
    'Erro (%)': np.abs((y_real.values - y_previsto) / y_real.values) * 100
})

# Adicionar informações contextuais
comparacao['Hour'] = exemplos['Hour'].values
comparacao['is_peak'] = exemplos['is_peak_operational'].values

print("\n📊 COMPARAÇÃO REAL vs PREVISTO (10 exemplos):\n")
print(comparacao.to_string(index=False))

print("\n📈 ESTATÍSTICAS DOS ERROS:")
print(f"   Erro absoluto médio: {comparacao['Erro (kWh)'].abs().mean():.2f} kWh")
print(f"   Maior erro: {comparacao['Erro (kWh)'].abs().max():.2f} kWh")

🎯 PREVISÕES EM EXEMPLOS REAIS

📊 COMPARAÇÃO REAL vs PREVISTO (10 exemplos):

 Real (kWh)  Previsto (kWh)    Erro (kWh)     Erro (%)  Hour  is_peak
       2.84        2.844300 -4.300000e-03 1.514085e-01    23        0
       2.95        3.381668 -4.316676e-01 1.463280e+01    14        1
      90.58       89.515500  1.064500e+00 1.175204e+00     9        1
      98.53       98.623000 -9.300000e-02 9.438750e-02     9        1
      95.22       95.428700 -2.087000e-01 2.191766e-01    23        0
       2.70        2.700000  4.884981e-15 1.809252e-13     4        0
       3.71        3.714500 -4.500000e-03 1.212938e-01     1        0
      89.57       89.375000  1.950000e-01 2.177068e-01     0        0
       3.85        3.848700  1.300000e-03 3.376623e-02     7        0
     105.37      105.892500 -5.225000e-01 4.958717e-01    23        0

📈 ESTATÍSTICAS DOS ERROS:
   Erro absoluto médio: 0.25 kWh
   Maior erro: 1.06 kWh


In [15]:
# Ver a ordem EXATA das colunas do treino
print("Colunas esperadas pelo modelo:")
print(X_train.columns.tolist())

Colunas esperadas pelo modelo:
['Lagging_Current_Reactive.Power_kVarh', 'Lagging_Current_Power_Factor', 'Hour', 'Month', 'DayOfMonth', 'is_peak_operational', 'hour_sin', 'hour_cos', 'is_weekend', 'load_type_encoded']


In [16]:
print("\n" + "="*70)
print("💰 SIMULAÇÃO DE ECONOMIA (E SE...?)")
print("="*70)

print("\n📋 CENÁRIO:")
print("   Uma carga tipo 'Maximum_Load' está programada para 10h")
print("   Queremos saber: 'E se eu mover para 3h (madrugada)?'")

# Criar dois cenários idênticos, exceto pela hora
cenario_pico = pd.DataFrame({
    'Lagging_Current_Reactive.Power_kVarh': [20.0],
    'Lagging_Current_Power_Factor': [85.0],
    'Hour': [10],  # ← PICO
    'Month': [6],
    'DayOfMonth': [15],
    'is_peak_operational': [1],  # ← É PICO
    'hour_sin': [np.sin(2 * np.pi * 10 / 24)],
    'hour_cos': [np.cos(2 * np.pi * 10 / 24)],
    'is_weekend': [0],
    'load_type_encoded': [59.3],  # Maximum
    
})

cenario_madrugada = cenario_pico.copy()
cenario_madrugada['Hour'] = [3]  # ← MADRUGADA
cenario_madrugada['is_peak_operational'] = [0]  # ← NÃO é pico
cenario_madrugada['hour_sin'] = [np.sin(2 * np.pi * 3 / 24)]
cenario_madrugada['hour_cos'] = [np.cos(2 * np.pi * 3 / 24)]

# Prever consumo
consumo_pico = model_rf.predict(cenario_pico)[0]
consumo_madrugada = model_rf.predict(cenario_madrugada)[0]

print(f"\n📊 PREVISÕES:")
print(f"   Cenário PICO (10h):        {consumo_pico:.2f} kWh")
print(f"   Cenário MADRUGADA (3h):    {consumo_madrugada:.2f} kWh")
print(f"   Diferença:                 {consumo_pico - consumo_madrugada:.2f} kWh")

# Calcular economia financeira
# Tarifa de pico: R$ 0.95/kWh | Madrugada: R$ 0.35/kWh
custo_pico = consumo_pico * 0.95
custo_madrugada = consumo_madrugada * 0.35
economia = custo_pico - custo_madrugada

print(f"\n💰 IMPACTO FINANCEIRO (por operação):")
print(f"   Custo PICO (R$ 0.95/kWh):       R$ {custo_pico:.2f}")
print(f"   Custo MADRUGADA (R$ 0.35/kWh):  R$ {custo_madrugada:.2f}")
print(f"   ECONOMIA:                       R$ {economia:.2f}")

# Projetar anual (assumindo 250 dias úteis/ano)
economia_anual = economia * 250
print(f"\n📈 PROJEÇÃO ANUAL (250 operações/ano):")
print(f"   Economia estimada: R$ {economia_anual:,.2f}/ano")

print("\n✅ Modelo permite SIMULAÇÕES de decisões operacionais!")


💰 SIMULAÇÃO DE ECONOMIA (E SE...?)

📋 CENÁRIO:
   Uma carga tipo 'Maximum_Load' está programada para 10h
   Queremos saber: 'E se eu mover para 3h (madrugada)?'

📊 PREVISÕES:
   Cenário PICO (10h):        33.46 kWh
   Cenário MADRUGADA (3h):    33.43 kWh
   Diferença:                 0.03 kWh

💰 IMPACTO FINANCEIRO (por operação):
   Custo PICO (R$ 0.95/kWh):       R$ 31.79
   Custo MADRUGADA (R$ 0.35/kWh):  R$ 11.70
   ECONOMIA:                       R$ 20.09

📈 PROJEÇÃO ANUAL (250 operações/ano):
   Economia estimada: R$ 5,022.34/ano

✅ Modelo permite SIMULAÇÕES de decisões operacionais!
